# Ejercicio 2 — Clasificación con EfficientNetB0

Clasificación binaria: `0 = sin paneles`, `1 = con paneles`. La etiqueta se deriva de la presencia de al menos una caja YOLO.

En Colab, el setup monta Google Drive y guarda los modelos y métricas en `Mi unidad/PDI-TP/resultados/<integrante>/`. Crear esa carpeta compartida, agregarla como acceso directo en `Mi unidad` y completar `INTEGRANTE` en la celda de setup.

Se comparan tres configuraciones de la misma red:
1. Pesos ImageNet con extractor congelado.
2. Pesos ImageNet con fine-tuning del extractor.
3. Pesos aleatorios (`weights=None`).

La clase positiva se define por la presencia de anotaciones, no por el valor de la clase YOLO. Confirmar con el equipo el significado de las clases 0, 1 y 2 antes de cerrar el informe.

In [ ]:
# Setup
INTEGRANTE = 'nombre'  # cambiar por tu nombre o identificador
import os, sys, json, random
from pathlib import Path

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

if 'google.colab' in sys.modules:
    REPO = '/content/PDI-TP'
    if os.path.isdir(REPO):
        !git -C {REPO} pull -q
    else:
        !git clone -q https://github.com/candeperles/PDI-TP.git {REPO}
else:
    REPO = str(Path.cwd().parent)

sys.path.insert(0, REPO)
from pdi_tp import entorno, datos, resultados
rutas = entorno.preparar(INTEGRANTE)
print('Repositorio:', REPO)
print('Imágenes:', rutas.imagenes)

In [ ]:
# Dependencias
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Split y etiquetas binarias
def nombres_split(path):
    return [Path(x.strip()).name for x in Path(path).read_text().splitlines() if x.strip()]

def etiqueta(nombre):
    label = rutas.labels / f'{Path(nombre).stem}.txt'
    return int(label.read_text().strip() != '')

def preparar_lista(split_path):
    nombres = nombres_split(split_path)
    paths = [str(rutas.imagenes / nombre) for nombre in nombres]
    labels = [etiqueta(nombre) for nombre in nombres]
    return np.array(paths), np.array(labels, dtype=np.float32)

train_paths, y_train = preparar_lista(Path(REPO) / 'splits' / 'train.txt')
val_paths, y_val = preparar_lista(Path(REPO) / 'splits' / 'val.txt')
print('Train:', len(train_paths), 'positivos:', int(y_train.sum()))
print('Val:', len(val_paths), 'positivos:', int(y_val.sum()))
print('Pesos de clase:', dict(zip(*np.unique(y_train, return_counts=True))))

In [ ]:
# tf.data
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def cargar(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
], name='augmentation')

def dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(cargar, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = dataset(train_paths, y_train, training=True)
val_ds = dataset(val_paths, y_val)

In [ ]:
# Modelo EfficientNetB0
def construir_modelo(weights='imagenet', congelado=True):
    base = keras.applications.EfficientNetB0(
        include_top=False, weights=weights, input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = not congelado
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = augment(inputs)
    # EfficientNet incluye su propio preprocesamiento.
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs, outputs), base

def entrenar(nombre, weights, fine_tune=False, epochs_head=10, epochs_ft=10):
    congelado = weights == 'imagenet' and not fine_tune
    model, base = construir_modelo(weights=weights, congelado=congelado)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs_head, callbacks=callbacks)

    if fine_tune and weights == 'imagenet':
        # Segunda etapa: fine-tuning con learning rate bajo.
        base.trainable = True
        model.compile(optimizer=keras.optimizers.Adam(1e-5),
                      loss='binary_crossentropy', metrics=['accuracy'])
        history_ft = model.fit(train_ds, validation_data=val_ds, epochs=epochs_ft, callbacks=callbacks)
        for key, values in history_ft.history.items():
            history.history.setdefault(key, []).extend(values)
    return model, history

## Entrenamiento

Ejecutar las variantes por separado si la memoria de la GPU es limitada. Los nombres quedan listos para comparar con MobileNet.

In [ ]:
experimentos = {
    "efficientnet_imagenet_congelado": ('imagenet', False),
    "efficientnet_imagenet_finetuning": ('imagenet', True),
    "efficientnet_aleatorio": (None, False),
}

modelos = {}
historias = {}
for nombre, (weights, congelado) in experimentos.items():
    print('\n===', nombre, '===')
    # Para el experimento de fine-tuning, entrenar() hace ambas etapas.
    modelo, historia = entrenar(nombre, weights, fine_tune=congelado, epochs_head=10, epochs_ft=10)
    modelos[nombre] = modelo
    historias[nombre] = historia.history
    modelo.save(rutas.resultados / f'{nombre}.keras')

# Si se quiere ejecutar solo una variante, comentar el bucle y usar, por ejemplo:
# modelo, historia = entrenar('efficientnet_imagenet_congelado', 'imagenet', fine_tune=False)

In [ ]:
# Métricas sobre validación
def evaluar(nombre, model):
    prob = model.predict(val_ds, verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)
    metricas = {
        'accuracy': float(accuracy_score(y_val, pred)),
        'precision': float(precision_score(y_val, pred, zero_division=0)),
        'recall': float(recall_score(y_val, pred, zero_division=0)),
        'f1': float(f1_score(y_val, pred, zero_division=0)),
        'umbral': 0.5,
        'matriz_confusion': confusion_matrix(y_val, pred).tolist(),
    }
    print(nombre)
    print(classification_report(y_val, pred, target_names=['sin_paneles', 'con_paneles'], zero_division=0))
    print('Matriz de confusión:\n', np.array(metricas['matriz_confusion']))
    resultados.guardar_experimento(
        rutas, nombre=nombre,
        params={'red': 'EfficientNetB0', 'weights': experimentos[nombre][0],
                'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE,
                'augmentation': True, 'umbral': 0.5},
        metricas=metricas
    )
    return metricas

metricas_efficientnet = {nombre: evaluar(nombre, model) for nombre, model in modelos.items()}
metricas_efficientnet

In [ ]:
# Curvas de entrenamiento
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for nombre, h in historias.items():
    axes[0].plot(h['loss'], label=nombre)
    axes[1].plot(h['val_accuracy'], label=nombre)
axes[0].set_title('Loss'); axes[0].set_xlabel('Época'); axes[0].legend()
axes[1].set_title('Accuracy de validación'); axes[1].set_xlabel('Época'); axes[1].legend()
plt.show()